In [1]:
# Install libraries 
!pip install seaborn --quiet
!pip install missingno --quiet
!pip install imblearn --quiet
!pip install scikit-learn --quiet

In [2]:
# Import required libraries 
import os 
import sys
import pandas as pd 
import seaborn as sns
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
import warnings

sys.path.append(os.path.abspath(".."))

# Import functions 
import functions.wrangling as wrg
import functions.missing_labs as ml
import functions.eda as eda
import functions.eda_model as em

warnings.filterwarnings("ignore")

# Set working directory (change this to the folder on your system)
os.chdir(r"G:\.shortcut-targets-by-id\1qO0AfYMqzVbXreDMm-gZUvrYXtVZCnDA\CHL8010F2  CPCSSN Dataset") 

In [3]:
# Load and clean datasets

# Define and load file paths 
file_paths = {
    'patient': 'C4MPatient.csv',
    'lab': 'C4MLab.csv',
    'diag': 'C4MEncounterdiagnosis.csv',
    'condition': 'C4MHealthCondition.csv'
}
datasets = wrg.load_csv(file_paths)

# Specify columns to keep from each dataset
columns_to_keep = {
    'patient': ["Patient_ID", "Sex", "BirthYear"],
    'lab': ["Patient_ID", "Name_calc", "TestResult_calc", "PerformedDate"],
    'diag': ["Patient_ID", "DiagnosisText_calc", "DiagnosisCode_calc", "DateCreated"],
    'condition': ["Patient_ID", "DiagnosisText_calc", "DateCreated"]
}
datasets = wrg.select_columns(datasets, columns_to_keep)

# Clean diagnosis data 
diagnosis_cleaning_steps = [
    ('DiagnosisText_calc', 'uppercase'),
    ('DiagnosisCode_calc', 'strip'),
    ('DiagnosisCode_calc', 'dropna'),
    ('DateCreated', 'datetime')
]
datasets['diag'] = wrg.replace_string_nan(datasets['diag'], 'DiagnosisCode_calc')
datasets['diag'] = wrg.preprocess_data(datasets['diag'], diagnosis_cleaning_steps)

In [8]:
# Extract bipolar disorder lab results and summarize patient counts

# Define BD ICD-9 codes and relevant markers
bd_codes = ["296.0", "296.1", "296.4", "296.5", "296.6", "296.7", "296.80", "296.89"]
relevant_markers = ["TOTAL CHOLESTEROL", "HBA1C", "HDL", "FASTING GLUCOSE", "LDL", "INR", "GLUCOSE TOLERANCE"]

# Extract lab results that occur after BD diagnosis (filtered by relevant markers)
bd_labs_after = wrg.extract_labs_relative_to_diagnosis(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    lab_test_names=relevant_markers
)

# Get the first BD diagnosis date per patient
first_dx = wrg.get_first_matching_diagnosis(
    df=datasets['diag'],
    diagnosis_col='DiagnosisCode_calc',
    date_col='DateCreated',
    target_codes=bd_codes,
    new_date_col='BD_Diagnosis_Date',
    new_code_col='BD_Code'
)

# Merge diagnosis info into labs 
bd_labs_after = bd_labs_after.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date', 'BD_Code']],
    on='Patient_ID', how='left'
).drop(columns=['Lab_Timing'])

# Pivot lab data to wide format
bd_labs_after = wrg.pivot_lab_data(
    bd_labs_after,
    index_cols=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    name_col='Name_calc',
    value_col='TestResult_calc'
).sort_values(['Patient_ID', 'PerformedDate'])

# Add age and sex data for each patient
bd_labs_after = wrg.add_demographics_to_labs(
    lab_df=bd_labs_after,
    patient_df=datasets['patient']
)

# Classify patients by lab timing using only relevant markers
lab_timing_summary, bd_first_clean = wrg.classify_lab_timing(
    lab_df=datasets['lab'][datasets['lab']['Name_calc'].isin(relevant_markers)],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    relevant_markers=None
)

# Count patients with non-BD same-day diagnoses AND lab results after BD diagnosis
nonbd_diag_same_day_after = wrg.non_bd_same_day_with_labs_after(
    diag_df=datasets['diag'],
    bd_labs_after_df=bd_labs_after,
    bd_first_clean=bd_first_clean,
    bd_codes=bd_codes
)

summary_stats = [
    ("Total BD patients (first-ever diagnosis)", bd_first_clean['Patient_ID'].nunique()),
    ("Patients with labs AFTER BD diagnosis", bd_labs_after['Patient_ID'].nunique()),
    ("Patients with lab AFTER BD diagnosis and non-BD diagnosis on same day", nonbd_diag_same_day_after)
]
wrg.print_summary_stats(summary_stats)

bd_labs_after.to_csv("bd_labs_after.csv",index=False)

Total BD patients (first-ever diagnosis): 378
Patients with labs AFTER BD diagnosis: 214
Patients with lab AFTER BD diagnosis and non-BD diagnosis on same day: 71


In [9]:
# Column selection and imputation 

# Load bd_labs_after dataset
df = ml.load_data("bd_labs_after.csv")

# define metadata columns
meta_cols = [
    "Patient_ID", "PerformedDate", "BD_Diagnosis_Date", "BD_Code"
]

# Get lab columns 
lab_cols = ml.get_lab_columns(df, meta_cols)

# Keep all lab columns and impute values for all of them
full_df, full_dropped = ml.impute_labs(
    df, meta_cols, lab_cols, 
    save_prefix="bd_labs_after_cleaned_all", 
    drop_missing=False
)

Loaded dataset with shape: (1196, 14)
Auto-detected lab columns: ['FASTING GLUCOSE', 'GLUCOSE TOLERANCE', 'HBA1C', 'HDL', 'INR', 'LDL', 'TOTAL CHOLESTEROL', 'Sex', 'BirthYear', 'Age']
Keeping all lab columns regardless of missingness.
Imputing FASTING GLUCOSE using median (skewness=3.98)
Imputing GLUCOSE TOLERANCE using mean (skewness=nan)
Imputing HBA1C using median (skewness=2.27)
Imputing HDL using median (skewness=1.06)
Imputing INR using mean (skewness=0.62)
Imputing LDL using mean (skewness=0.11)
Imputing TOTAL CHOLESTEROL using mean (skewness=-0.00)


TypeError: could not convert string to float: 'Female'

In [ ]:
# Load imputed, clean dataset 
df_all_vars =  ml.load_data("bd_labs_after_cleaned_all.csv")

# Get concise summary of the structure and content of DataFrames
eda.summarize_dataframes(df_all_vars)

# Calculate summary statistics for lab markers
lab_cols_all = ['TOTAL CHOLESTEROL', 'FASTING GLUCOSE', 'HBA1C', 'HDL', 'LDL', 'INR', 'GLUCOSE TOLERANCE']
numeric_columns = {
    'all_vars': lab_cols_all
}
summaries = eda.summarize_numeric_statistics(df_all_vars, numeric_columns)

# Plot histograms for lab markers (all_vars dataset, default 10 bins)
print("Histogram with 10 bins")
eda.plot_histograms(df_all_vars, lab_cols_all)

# Plot correlation matrix for all lab columns
eda.corr_matrix = eda.plot_correlation_matrix(df_all_vars, lab_cols_all)

# Correlation matrix excluding GLUCOSE TOLERANCE
lab_cols_reduced = ['TOTAL CHOLESTEROL', 'FASTING GLUCOSE', 'HBA1C', 'HDL', 'LDL', 'INR']
corr_matrix_reduced = eda.plot_correlation_matrix(df_all_vars, lab_cols_reduced)

# Check why GLUCOSE TOLERANCE has NaN correlations
eda.check_unique_values(df_all_vars, 'GLUCOSE TOLERANCE')

# Plot boxplots to analyze outliers
eda.plot_boxplots(df_all_vars, lab_cols_reduced)

# Plot scatter pairs for high-correlation combinations
eda.plot_scatter_pairs(df_all_vars, [
    ('TOTAL CHOLESTEROL', 'LDL', 'Total Cholesterol vs LDL'),
    ('FASTING GLUCOSE', 'HBA1C', 'Fasting Glucose vs HBA1C')
])